In [1]:
import os

for root, dirs, files in os.walk("/content"):
    if "colab_scaffold.py" in files:
        print(os.path.join(root, "colab_scaffold.py"))

/content/colab_scaffold.py


In [2]:
exec(open("/content/colab_scaffold.py").read())

installing: transformers==4.46.* accelerate==1.1.*
profiling pins installed (no vLLM today)
installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*


KeyboardInterrupt: 

Closed the above cell because i don't need the whole file just the libraries to install

In [15]:

import subprocess, sys

# Pins mirrored from ../../../PINS.md. Keep these two in sync (PINS.md wins).
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

# %%
# INSTALL CELL A: profiling only, NO SERVER. This is day 1.
#
# Day 1 loads the model with transformers and reads the card. It never serves,
# so it must NOT install vLLM. Installing vLLM here would replace Colab's torch
# with vLLM's older build AND downgrade numpy to 1.26, and Colab's preinstalled
# extensions are compiled against numpy 2. The model load then dies with
#   RuntimeError: Failed to import transformers.models.qwen2.modeling_qwen2
#   ... numpy.dtype size changed, Expected 96 from C header, got 88
# Verified on a T4, 2026-07-27. Colab's own torch is the torch today. This
# install is about two minutes, not thirty.
pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
)
print("profiling pins installed (no vLLM today)")

installing: transformers==4.46.* accelerate==1.1.*
profiling pins installed (no vLLM today)


In [24]:
import torch
import transformers
import accelerate
import numpy

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("numpy:", numpy.__version__)

torch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
transformers: 4.46.3
accelerate: 1.1.1
numpy: 2.1.3


In [26]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

In [18]:
import time, threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tok, skip_prompt=True,
                                    skip_special_tokens=True)
    kwargs = dict(**enc, max_new_tokens=new_tokens, do_sample=False,
                  streamer=streamer)
    th = threading.Thread(target=model.generate, kwargs=kwargs)
    t0 = time.time()
    th.start()
    stamps = []
    for _ in streamer:
        stamps.append(time.time())
    th.join()
    ttft = stamps[0] - t0
    # mean inter-token gap over the tokens after the first
    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0
    total = stamps[-1] - t0
    return {"ttft_s": round(ttft, 4), "tpot_s": round(tpot, 4),
            "total_s": round(total, 4), "n_tokens": len(stamps)}

# Warm-up, and it is not optional. The first generation on a fresh runtime pays
# CUDA context init and kernel autotuning, and all of that lands inside its TTFT.
# Time it and the shortest prompt comes out slowest, which is backwards and would
# tell you prefill does not depend on prompt length. Throw one generation away.
measure_stream(prompt_of_len(128), new_tokens=8)

ttft_by_len = {}
for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)

128 {'ttft_s': 0.0385, 'tpot_s': 0.0679, 'total_s': 8.7311, 'n_tokens': 129}
512 {'ttft_s': 0.1016, 'tpot_s': 0.0691, 'total_s': 8.944, 'n_tokens': 129}
2048 {'ttft_s': 0.3798, 'tpot_s': 0.0642, 'total_s': 8.5994, 'n_tokens': 129}


In [27]:
import json

with open("batch_check.json", "w") as f:
    json.dump({
        "batch1_tokens_per_s": batch_rows["1"],
        "batch8_tokens_per_s": batch_rows["8"]
    }, f, indent=2)

print("batch_check.json written")

batch_check.json written


In [28]:
import gc

def kv_formula_kb_per_token(layers=28, kv_heads=2, head_dim=128, dbytes=2):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024  # 28.0 KB

def cache_bytes(pkv):
    """Bytes the KV cache itself holds, read straight off the cache tensors."""
    if hasattr(pkv, "key_cache"):        # transformers returns a Cache object
        tensors = list(pkv.key_cache) + list(pkv.value_cache)
    else:                                # legacy tuple of (k, v) per layer
        tensors = [t for layer in pkv for t in layer]
    return sum(t.numel() * t.element_size() for t in tensors)

def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    enc = tok(prompt_of_len(context), return_tensors="pt").to("cuda")
    before = torch.cuda.memory_allocated()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False,
                         use_cache=True, return_dict_in_generate=True)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    total_tokens = out.sequences.shape[1]  # prompt + generated, the full KV span
    return {
        "context": context,
        "total_tokens": int(total_tokens),
        # what the whole generation cost, cache and activations together
        "peak_kb_per_token": round((peak - before) / total_tokens / 1024, 1),
        # the cache on its own, which is what the formula predicts
        "kv_kb_per_token": round(cache_bytes(out.past_key_values) / total_tokens / 1024, 1),
    }

formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)
kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]
for r in kv_rows:
    print(r, "  vs formula", formula, "KB/token")

# the green check compares the cache itself, not the whole-generation peak
import json
with open("kv_check.json", "w") as f:
    json.dump({"formula_kb_per_token": formula,
               "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
               "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]}, f)

formula KB/token: 28.0
{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 63.4, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 84.0, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 87.6, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token


In [29]:
# 24 requests: 18 that want 32 tokens, 6 that want 256. 2112 useful tokens.
QUEUE = [32, 32, 32, 256] * 6

def static_queue(batch: int, prompt: str = "Explain what an inference server does."):
    """A server WITHOUT continuous batching: a batch starts, nothing new joins
    until every member has finished, so it runs until its SLOWEST member."""
    t0 = time.time(); useful = 0; slots = 0
    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]
        n = max(chunk)                    # the batch runs until the slowest
        enc = tok([prompt] * len(chunk), return_tensors="pt",
                  padding=True).to("cuda")
        model.generate(**enc, max_new_tokens=n, do_sample=False)
        useful += sum(chunk)              # tokens anyone actually asked for
        slots += n * len(chunk)           # token-slots the GPU actually decoded
        # accounting note: this counts REQUESTED tokens. Greedy decoding on
        # these prompts runs to the max_new_tokens cap, so requested equals
        # generated here; tomorrow's vLLM client counts the server's own
        # completion_tokens, and its README says so. Same convention, stated.
    dt = time.time() - t0
    return {"batch": batch, "wall_s": round(dt, 2),
            "tokens_per_s": round(useful / dt, 1),
            "slot_efficiency": round(useful / slots, 3)}

batch_rows = {}
for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)

{'batch': 1, 'wall_s': 62.96, 'tokens_per_s': 33.5, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 43.28, 'tokens_per_s': 48.8, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 22.09, 'tokens_per_s': 95.6, 'slot_efficiency': 0.344}


In [30]:
import json
baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,                    # by prompt length
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},  # tokens_per_s at 1,4,8
}
with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)
print(json.dumps(baselines, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.0385,
    "512": 0.1016,
    "2048": 0.3798
  },
  "tpot_s": 0.0343,
  "batch": {
    "1": 33.5,
    "4": 48.8,
    "8": 95.6
  }
}


In [31]:
from google.colab import files
files.download("baselines.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
# Green-check verifier for Lab W3D1 (profile inference).
# Paste this as the last cell of your day-1 notebook and run it. It reads
# profile.json (the matrix rows you wrote) and checks the schema and the sanity
# rules. It also reads the batch experiment numbers if you saved them to
# batch_check.json; if that file is absent it asks for the two numbers inline so
# the batch-8 > batch-1 rule can still be checked.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os

REQUIRED_KEYS = {"dtype", "context", "vram_gb", "util_mean", "tokens_per_s"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found; write it in the last data cell")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")


def main() -> None:
    rows = load_json("profile.json")

    if not isinstance(rows, list) or not rows:
        fail("profile.json must be a non-empty list of rows")

    # schema
    for i, row in enumerate(rows):
        if not isinstance(row, dict):
            fail(f"row {i} is not an object")
        missing = REQUIRED_KEYS - set(row)
        if missing:
            fail(f"row {i} missing keys: {sorted(missing)}")

    dtypes = {r["dtype"] for r in rows}
    contexts = sorted({r["context"] for r in rows})
    if "fp16" not in dtypes:
        fail("no fp16 rows; the matrix needs fp16")
    if len(contexts) < 3:
        fail(f"need at least 3 context lengths, found {contexts}")

    # sanity 1: VRAM rises with context (within each dtype)
    for dt in dtypes:
        sub = sorted((r for r in rows if r["dtype"] == dt),
                     key=lambda r: r["context"])
        vrams = [r["vram_gb"] for r in sub]
        if any(b < a - 0.01 for a, b in zip(vrams, vrams[1:])):
            fail(f"{dt} VRAM does not rise with context: {vrams}")

    # sanity 2: fp16 uses more memory than int8 at a shared context
    if "int8" in dtypes:
        shared = None
        for c in contexts:
            has_fp16 = any(r["dtype"] == "fp16" and r["context"] == c for r in rows)
            has_int8 = any(r["dtype"] == "int8" and r["context"] == c for r in rows)
            if has_fp16 and has_int8:
                shared = c
                break
        if shared is None:
            fail("fp16 and int8 share no context length to compare")
        fp16_v = next(r["vram_gb"] for r in rows
                      if r["dtype"] == "fp16" and r["context"] == shared)
        int8_v = next(r["vram_gb"] for r in rows
                      if r["dtype"] == "int8" and r["context"] == shared)
        if not fp16_v > int8_v:
            fail(f"fp16 VRAM ({fp16_v}) not above int8 VRAM ({int8_v}) at "
                 f"context {shared}")

    # sanity 3: batch-8 tokens/s beats batch-1
    b1 = b8 = None
    if os.path.exists("batch_check.json"):
        bc = load_json("batch_check.json")
        b1 = bc.get("batch1_tokens_per_s")
        b8 = bc.get("batch8_tokens_per_s")
    else:
        # allow the two numbers as module-level names set in an earlier cell
        b1 = globals().get("BATCH1_TOKENS_PER_S")
        b8 = globals().get("BATCH8_TOKENS_PER_S")
    if b1 is None or b8 is None:
        fail("batch numbers missing; save batch_check.json with "
             "batch1_tokens_per_s and batch8_tokens_per_s, or set "
             "BATCH1_TOKENS_PER_S / BATCH8_TOKENS_PER_S")
    if not b8 > b1:
        fail(f"batch-8 tokens/s ({b8}) not above batch-1 ({b1})")

    print(f"rows: {len(rows)}, dtypes: {sorted(dtypes)}, contexts: {contexts}")
    print(f"batch-1 tokens/s: {b1}, batch-8 tokens/s: {b8}")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 32.3, batch-8 tokens/s: 92.7
GREEN CHECK: PASS
